# Within Subject — pure DL, full retraining

Attach **only** `low-cost-motor-imagery-decoding-for-rehab` and run all cells from a fresh **GPU** kernel.
This notebook embeds its exact source and trains from the challenge's raw CSVs.
It requires PyTorch, NumPy, pandas and scikit-learn (used only to define Within
stratified splits); no installation, network, external data, cached feature or
pretrained checkpoint is required. This is a reproducible baseline candidate.

The model takes the eight named EEG channels in `[0,4.8)` at nominal 250 Hz.
Markers segment trials/provide training labels; timestamps check continuity;
filenames establish allowed partitions and submission order. The unchanged
PhaseConvNet normalizes each trial/channel at runtime, then learns temporal
and spatial convolutions and classification jointly. There are no classical
features, filtering, target-batch adaptation or forced class counts.

Each participant uses only their own supplied labels. Two independent inner stopping fits use that participant’s calibration data and disjoint run-3 support halves. The rounded median of the two stopping epochs determines a fresh fit on all their labels, exactly as in the research pipeline. No outer-query prediction or score is computed.

AdamW uses learning rate .001, weight decay .01, batch 64 and seed 20261003.
Within stopping allows 60 epochs, patience 10, then independently seeded refits.
Deterministic algorithms, cuDNN deterministic mode, cuDNN TF32 enabled and CUDA
matmul TF32 disabled are explicit. Immediate checkpoint replay must preserve
labels and meet probability `1e-6` / logit `1e-5` absolute tolerances.
An earlier L40S-to-CPU replay changed one borderline Within label; identity
across devices or PyTorch versions is not assumed. Run and validate on Kaggle's
actual GPU before submitting.

Outputs include `submission.csv`, ordered `test_predictions.csv`, `metadata.csv`,
source/input/output hashes and numerical settings in `manifest.json`, stopping
provenance, full training histories, final checkpoints and replay checks.
No competition upload is performed. The full CSV-generating notebook must
also execute in Kaggle before submission.


In [ ]:
import os
from pathlib import Path

COMPETITION = 'within_subject'
SLUG = 'low-cost-motor-imagery-decoding-for-rehab'
TEST_FOLDER = 'Single subject test set'
EXPECTED_TEST_ROWS = 680
EXPECTED_TEST_FILES = 17
EXPECTED_TEST_PEOPLE = 17
NOTEBOOK_NAME = 'within_subject_dl.ipynb'
BUILD_ID = 'e0566192bf51b0fda7f25d777035f4600667bc57997d4c5eadba9aa97c10a46f'
DATA_ROOT = os.environ.get('EEG_DATA_ROOT') or None
OUTPUT_DIR = Path(os.environ.get('EEG_OUTPUT_DIR', '/kaggle/working'))
SEED = int(os.environ.get('EEG_SEED', '20261003'))
THREADS = int(os.environ.get('EEG_THREADS', '4'))
DEVICE = 'cuda'
WINDOW = 'task0_48'
MAX_EPOCHS = 60
CROSS_EPOCHS = 19
if THREADS < 1:
    raise ValueError('THREADS must be positive')
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[variable] = str(THREADS)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any((OUTPUT_DIR / name).exists() for name in ['manifest.json', 'submission.csv', 'test_predictions.csv', 'final', 'inner']):
    raise FileExistsError('Use a fresh output directory; existing candidate files are preserved')


## Embedded source and numerical settings


In [ ]:
import argparse
import hashlib
import importlib
import json
import platform
import sys
import time
from dataclasses import asdict

STARTED = time.monotonic()
SOURCE_HASHES = {'eeg_comp/data.py': '40eebf639479ea6b34041b4c592c7e532b166e0db31b32973b3799df8b7e8c53', 'eeg_comp/neural.py': '7138f9e7a47d9b4a64fc73ced9a1e1b248f393bf182788d07ef00b9e7bf62ea7', 'eeg_comp/submission.py': '8d68409c4cc695229ee999aeab792fdd17829dcb32cc66d9428e0d66c3310473', 'scripts/run_neural.py': '07cfa79d38f28741913ce474874604a96f851be61558e8dde32485c04842d5c3', 'eeg_comp/__init__.py': 'c418c694b9115cdfbb88ffab570625247944861e6ee635e115aa621b111178a7', 'scripts/__init__.py': '64127ae091eefcec270e4a84c2ba214e48770850959d880b703992a2bd21e094'}
SOURCES = {'eeg_comp/data.py': '"""Marker-led loading with an explicit native-sample epoch contract.\n\nOnly named EEG columns enter X. Time, marker spellings and file metadata are\naudit/segmentation information, never predictor columns. Each competition is\nread independently; no data matching across competitions is implemented.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nCHANNELS = ("Fz", "C3", "Cz", "C4", "PO7", "Pz", "PO8", "Oz")\nFS = 250\nCUE_INDEX = 750\nEPOCH_SAMPLES = 2000\nLABELS = {"rest": 0, "move": 1}\nCOMPETITIONS = ("within_subject", "cross_subject")\nSTART_MARKERS = {"trial_start", "trail_start"}\nKNOWN_MARKERS = START_MARKERS | {"rest", "move", "cue_start", "cue_stop", "exp_start", "exp_stop"}\n\n\n@dataclass(frozen=True)\nclass Trial:\n    trial_start: int\n    cue: int\n    stop: int\n    label: str\n\n\ndef parse_markers(markers, split: str) -> tuple[list[Trial], dict]:\n    """Pair each cue with its own fixation start and first subsequent cue_stop.\n\n    Extra stop markers outside a trial are reported, not converted into trials.\n    Missing boundaries, multiple cues, mixed labeled/test cues and unknown\n    marker types are rejected rather than silently losing or relabeling trials.\n    """\n    if split not in {"train", "test"}:\n        raise ValueError(f"Invalid split: {split}")\n    trials: list[Trial] = []\n    counts: Counter = Counter()\n    orphan_stops, unused_starts = [], []\n    start = cue = None\n    label = ""\n    for row, value in enumerate(markers):\n        if pd.isna(value):\n            continue\n        marker = str(value).strip().lower()\n        if not marker:\n            continue\n        counts[marker] += 1\n        if marker not in KNOWN_MARKERS:\n            raise ValueError(f"Unknown marker {marker!r} at row {row}")\n        if marker in START_MARKERS:\n            if cue is not None:\n                raise ValueError(f"Missing cue_stop before trial start at row {row}")\n            if start is not None:\n                unused_starts.append(start)\n            start = row\n        elif marker in {"rest", "move", "cue_start"}:\n            if (split == "train") != (marker in LABELS):\n                raise ValueError(f"Unexpected {marker} in {split} at row {row}")\n            if cue is not None:\n                raise ValueError(f"Multiple cues before cue_stop at row {row}")\n            if start is None:\n                raise ValueError(f"Cue without trial start at row {row}")\n            cue, label = row, marker if split == "train" else ""\n        elif marker == "cue_stop":\n            if cue is None:\n                orphan_stops.append(row)\n            else:\n                trials.append(Trial(start, cue, row, label))\n                start = cue = None\n        elif marker == "exp_stop":\n            if cue is not None:\n                raise ValueError(f"Experiment ended before cue_stop at row {row}")\n            if start is not None:\n                unused_starts.append(start)\n                start = None\n        elif marker == "exp_start" and (start is not None or cue is not None):\n            raise ValueError(f"Experiment restarted inside a trial at row {row}")\n    if cue is not None:\n        raise ValueError("File ended with an unclosed cue")\n    if start is not None:\n        unused_starts.append(start)\n    if not trials:\n        raise ValueError("No complete trials found")\n    return trials, {"marker_counts": dict(counts), "orphan_stop_rows": orphan_stops,\n                    "unused_start_rows": unused_starts}\n\n\ndef identify_file(path: Path, competition: str, split: str) -> tuple[str, int]:\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    patterns = {\n        "train": r"(S\\d{3})-(\\d{3})_eeg\\.csv",\n        "cross_subject": r"(S\\d{3})_test_(\\d{3})_eeg\\.csv",\n        "within_subject": r"(S\\d{3})_test\\.csv",\n    }\n    match = re.fullmatch(patterns["train" if split == "train" else competition], path.name)\n    if match is None:\n        raise ValueError(f"Unexpected EEG file name: {path.name}")\n    return match[1], int(match[2]) if len(match.groups()) == 2 else 3\n\n\ndef extract_epochs(frame: pd.DataFrame, trials: list[Trial]) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[dict], dict]:\n    """Extract [-750,1250) native rows around cue; no temporal resampling.\n\n    X contains real recorded samples outside the exact fixation/task bounds,\n    flagged separately by phase_valid_mask. Missing/nonfinite samples and\n    samples across a timestamp discontinuity from the cue are NaN, with\n    valid_mask=False. A discontinuity is <=0 or >1.5 * median positive dt.\n    Phase boundaries are half-open: baseline starts at trial_start and task\n    ends immediately before cue_stop. Full native phase lengths are retained.\n    """\n    required = {"time", "Marker_val", *CHANNELS}\n    missing = required - set(frame.columns)\n    if missing:\n        raise ValueError(f"Missing columns: {sorted(missing)}")\n    time = frame["time"].to_numpy(dtype=np.float64)\n    eeg = frame.loc[:, CHANNELS].to_numpy(dtype=np.float32)\n    if len(time) < 2 or not np.isfinite(time).all():\n        raise ValueError("Recording must have at least two finite timestamps")\n    dt = np.diff(time)\n    positive = dt[dt > 0]\n    if not len(positive):\n        raise ValueError("Recording timestamps never increase")\n    median_dt = float(np.median(positive))\n    if not 0.0035 < median_dt < 0.0045:\n        raise ValueError(f"Unexpected sampling interval: {median_dt}")\n    discontinuity = (dt <= 0) | (dt > 1.5 * median_dt)\n    segments = np.r_[0, np.cumsum(discontinuity)]\n    finite_rows = np.isfinite(eeg).all(axis=1)\n    X = np.full((len(trials), len(CHANNELS), EPOCH_SAMPLES), np.nan, dtype=np.float32)\n    valid = np.zeros((len(trials), EPOCH_SAMPLES), dtype=bool)\n    phase_valid = valid.copy()\n    metadata = []\n    offsets = np.arange(-CUE_INDEX, EPOCH_SAMPLES - CUE_INDEX)\n    for i, trial in enumerate(trials):\n        if not 0 <= trial.trial_start < trial.cue < trial.stop < len(frame):\n            raise ValueError(f"Invalid native trial boundaries: {trial}")\n        rows = trial.cue + offsets\n        inside = (rows >= 0) & (rows < len(frame))\n        idx = np.flatnonzero(inside)\n        valid[i, idx] = finite_rows[rows[idx]] & (segments[rows[idx]] == segments[trial.cue])\n        good = np.flatnonzero(valid[i])\n        X[i][:, good] = eeg[rows[good]].T\n        phase_valid[i] = valid[i] & (rows >= trial.trial_start) & (rows < trial.stop)\n        metadata.append({\n            "trial_index": i, "trial_start_row": trial.trial_start,\n            "cue_row": trial.cue, "cue_stop_row": trial.stop,\n            "trial_start_time": float(time[trial.trial_start]),\n            "cue_time": float(time[trial.cue]), "cue_stop_time": float(time[trial.stop]),\n            "label": trial.label, "y": LABELS.get(trial.label, -1),\n            "baseline_samples": trial.cue - trial.trial_start,\n            "native_cue_samples": trial.stop - trial.cue,\n            "baseline_seconds": float(time[trial.cue] - time[trial.trial_start]),\n            "cue_seconds": float(time[trial.stop] - time[trial.cue]),\n            "valid_samples": int(valid[i].sum()),\n            "phase_valid_samples": int(phase_valid[i].sum()),\n            "baseline_2s_valid": bool(phase_valid[i, CUE_INDEX - 500:CUE_INDEX].all()),\n            "task_4p8s_valid": bool(phase_valid[i, CUE_INDEX:CUE_INDEX + 1200].all()),\n            "task_5s_valid": bool(phase_valid[i, CUE_INDEX:].all()),\n        })\n    timing = {\n        "median_dt_seconds": median_dt, "effective_fs_hz": 1.0 / median_dt,\n        "dt_min_seconds": float(dt.min()), "dt_max_seconds": float(dt.max()),\n        "dt_std_seconds": float(dt.std()), "discontinuity_rows": (np.flatnonzero(discontinuity) + 1).tolist(),\n        "nonpositive_intervals": int((dt <= 0).sum()),\n        "gap_intervals": int((dt > 1.5 * median_dt).sum()),\n        "max_abs_dt_deviation_seconds": float(np.max(np.abs(dt - median_dt))),\n        "first_time": float(time[0]), "last_time": float(time[-1]),\n    }\n    return X, valid, phase_valid, metadata, timing\n\n\ndef channel_quality(frame: pd.DataFrame) -> dict:\n    """Descriptive quality statistics in file units; no artifact cleaning."""\n    result = {}\n    for channel in CHANNELS:\n        values = frame[channel].to_numpy(dtype=np.float64)\n        finite = values[np.isfinite(values)]\n        if not len(finite):\n            result[channel] = {"finite_fraction": 0.0}\n            continue\n        diff = np.diff(values)\n        equal = diff == 0\n        edges = np.diff(np.r_[False, equal, False].astype(int))\n        starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)\n        q = np.quantile(finite, [.001, .01, .5, .99, .999])\n        result[channel] = {\n            "finite_fraction": float(len(finite) / len(values)),\n            "min": float(finite.min()), "max": float(finite.max()),\n            "q001": float(q[0]), "q01": float(q[1]), "median": float(q[2]),\n            "q99": float(q[3]), "q999": float(q[4]), "std": float(finite.std()),\n            "max_abs_step": float(np.nanmax(np.abs(diff))),\n            "equal_adjacent_fraction": float(equal.mean()),\n            "longest_constant_samples": int((stops - starts).max() + 1) if len(starts) else 1,\n            "fraction_at_min": float(np.mean(values == finite.min())),\n            "fraction_at_max": float(np.mean(values == finite.max())),\n        }\n    return result\n\n\ndef competition_paths(data_root: Path, competition: str) -> list[tuple[str, Path]]:\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    root = data_root / competition\n    test_folder = "Single subject test set" if competition == "within_subject" else "Cross participant test set"\n    paths = [(split, p) for split, folder in [("train", "Training set"), ("test", test_folder)]\n             for p in sorted((root / folder).glob("*.csv"))]\n    if {split for split, _ in paths} != {"train", "test"}:\n        raise FileNotFoundError(f"Missing train/test files under {root}")\n    return paths\n\n\ndef audit_competition(data_root: Path, output_root: Path, competition: str) -> dict:\n    """Read and write exactly one competition; dictionaries never span tasks."""\n    paths = competition_paths(data_root, competition)\n    arrays, masks, phase_masks, metadata, file_reports = [], [], [], [], []\n    file_hashes, epoch_hashes = {}, {}\n    duplicate_files, duplicate_epochs = [], []\n    test_order = 0\n    for split, path in paths:\n        subject, run = identify_file(path, competition, split)\n        frame = pd.read_csv(path)\n        trials, marker_report = parse_markers(frame["Marker_val"], split)\n        X, valid, phase_valid, rows, timing = extract_epochs(frame, trials)\n        digest = hashlib.sha256(path.read_bytes()).hexdigest()\n        if digest in file_hashes:\n            duplicate_files.append([file_hashes[digest], path.name])\n        file_hashes[digest] = path.name\n        for local_index, row in enumerate(rows):\n            row.update({"epoch_index": len(metadata), "competition": competition,\n                        "split": split, "subject": subject, "run": run,\n                        "file": path.relative_to(data_root / competition).as_posix(),\n                        "test_order": test_order if split == "test" else -1})\n            if split == "test":\n                test_order += 1\n            # Hash original float64 numeric values, before cache quantization.\n            # Compare recorded EEG only within this competition.\n            cue_rows = frame.iloc[trials[local_index].cue:trials[local_index].cue + 1000]\n            cue_digest = hashlib.sha256(cue_rows.loc[:, CHANNELS].to_numpy(dtype=np.float64).tobytes()).hexdigest()\n            reference = {"file": row["file"], "trial_index": row["trial_index"], "split": split}\n            if cue_digest in epoch_hashes:\n                duplicate_epochs.append([epoch_hashes[cue_digest], reference])\n            epoch_hashes[cue_digest] = reference\n            metadata.append(row)\n        file_reports.append({"file": path.relative_to(data_root / competition).as_posix(),\n                             "split": split, "subject": subject, "run": run,\n                             "rows": len(frame), "columns": frame.columns.tolist(), "sha256": digest,\n                             "n_trials": len(trials), "labels": dict(Counter(t.label for t in trials if t.label)),\n                             **marker_report, **timing, "channels": channel_quality(frame),\n                             "overlapping_native_trials": [i for i in range(1, len(trials))\n                                                           if trials[i].trial_start <= trials[i - 1].stop],\n                             "overlapping_cached_epochs": [i for i in range(1, len(trials))\n                                                            if trials[i].cue - trials[i - 1].cue < EPOCH_SAMPLES]})\n        arrays.append(X); masks.append(valid); phase_masks.append(phase_valid)\n        print(f"{competition} {path.name}: {len(trials)} trials", flush=True)\n    meta = pd.DataFrame(metadata)\n    X = np.concatenate(arrays)\n    valid = np.concatenate(masks)\n    phase_valid = np.concatenate(phase_masks)\n    chronology = []\n    for subject in sorted(meta.subject.unique()):\n        train3 = meta[(meta.subject == subject) & (meta.split == "train") & (meta.run == 3)]\n        test3 = meta[(meta.subject == subject) & (meta.split == "test") & (meta.run == 3)]\n        if len(train3) and len(test3):\n            train_report = next(r for r in file_reports if r["subject"] == subject and r["run"] == 3 and r["split"] == "train")\n            test_report = next(r for r in file_reports if r["subject"] == subject and r["run"] == 3 and r["split"] == "test")\n            join_dt = test_report["first_time"] - train_report["last_time"]\n            chronology.append({"subject": subject, "train_trials": len(train3), "test_trials": len(test3),\n                               "all_labeled_cues_before_test": bool(train3.cue_time.max() < test3.cue_time.min()),\n                               "file_join_dt_seconds": join_dt,\n                               "files_abut_one_sample": bool(abs(join_dt / train_report["median_dt_seconds"] - 1) < .01),\n                               "recording_time_ranges_overlap": bool(test_report["first_time"] <= train_report["last_time"]),\n                               "train_internal_discontinuities": len(train_report["discontinuity_rows"]),\n                               "test_internal_discontinuities": len(test_report["discontinuity_rows"]),\n                               "last_train_cue_time": float(train3.cue_time.max()),\n                               "first_test_cue_time": float(test3.cue_time.min()),\n                               "sum_observed_trials": len(train3) + len(test3)})\n    counts = meta.groupby(["split", "subject", "run"], sort=True).agg(\n        trials=("epoch_index", "size"), rest=("label", lambda s: (s == "rest").sum()),\n        move=("label", lambda s: (s == "move").sum())).reset_index()\n    report = {\n        "competition": competition, "cache_version": 1,\n        "channels": list(CHANNELS), "nominal_fs": FS,\n        "amplitude_units": "Unverified; CSV contains numeric raw device units with large DC offsets",\n        "acquisition_reference": "Not established from delivered CSVs",\n        "predictor_columns": list(CHANNELS), "files": file_reports,\n        "counts": counts.to_dict(orient="records"), "run3_chronology": chronology,\n        "exact_duplicate_files_within_competition": duplicate_files,\n        "exact_duplicate_4s_epochs_within_competition": duplicate_epochs,\n        "train_trials": int((meta.split == "train").sum()), "test_trials": int((meta.split == "test").sum()),\n        "trial_lengths": {col: {"min": float(meta[col].min()), "max": float(meta[col].max()),\n                                 "median": float(meta[col].median())}\n                          for col in ["baseline_samples", "native_cue_samples", "baseline_seconds", "cue_seconds"]},\n        "invalid_recorded_samples": int((~valid).sum()),\n        "invalid_phase_samples": int((~phase_valid).sum()),\n        "invalid_baseline_2s_trials": int((~meta.baseline_2s_valid).sum()),\n        "invalid_task_4p8s_trials": int((~meta.task_4p8s_valid).sum()),\n        "invalid_task_5s_trials": int((~meta.task_5s_valid).sum()),\n        "sample_submission_available": False,\n        "test_order_contract": "0-based, lexicographically sorted test filenames then chronological cue order",\n        "run_mixture": meta.groupby(["split", "run"]).size().rename("trials").reset_index().to_dict(orient="records"),\n        "subjects": {split: sorted(meta.loc[meta.split == split, "subject"].unique().tolist())\n                     for split in ["train", "test"]},\n    }\n    output = output_root / competition\n    output.mkdir(parents=True, exist_ok=True)\n    # Uncompressed cache favors repeatable short HPC reads over CPU compression.\n    np.savez(output / "epochs.npz", X=X, y=meta.y.to_numpy(dtype=np.int8),\n             valid_mask=valid, phase_valid_mask=phase_valid, fs=np.array(FS),\n             cue_index=np.array(CUE_INDEX), channels=np.array(CHANNELS), cache_version=np.array(1))\n    meta.to_csv(output / "metadata.csv", index=False)\n    counts.to_csv(output / "counts.csv", index=False)\n    (output / "audit.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\ndef load_cache(artifact_root: str | Path, competition: str):\n    """Load one competition and verify row/label/order alignment."""\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    root = Path(artifact_root) / competition\n    with np.load(root / "epochs.npz", allow_pickle=False) as cache:\n        arrays = {key: cache[key] for key in cache.files}\n    if int(arrays["cache_version"]) != 1 or int(arrays["fs"]) != FS or int(arrays["cue_index"]) != CUE_INDEX:\n        raise ValueError("Unsupported cache contract")\n    if arrays["X"].shape[1:] != (len(CHANNELS), EPOCH_SAMPLES) or arrays["X"].dtype != np.float32:\n        raise ValueError("Invalid epoch shape or dtype")\n    if tuple(arrays["channels"]) != CHANNELS:\n        raise ValueError("Invalid cached channel order")\n    for key in ["valid_mask", "phase_valid_mask"]:\n        if arrays[key].shape != (len(arrays["X"]), EPOCH_SAMPLES) or arrays[key].dtype != bool:\n            raise ValueError(f"Invalid {key}")\n    if (arrays["phase_valid_mask"] & ~arrays["valid_mask"]).any():\n        raise ValueError("Phase validity extends beyond recording validity")\n    metadata = pd.read_csv(root / "metadata.csv", keep_default_na=False)\n    if len(metadata) != len(arrays["X"]) or not np.array_equal(metadata.epoch_index, np.arange(len(metadata))):\n        raise ValueError("Metadata and epoch cache are not aligned")\n    if not (metadata.competition == competition).all() or not np.array_equal(metadata.y, arrays["y"]):\n        raise ValueError("Competition or label mismatch")\n    if not metadata.split.isin(["train", "test"]).all():\n        raise ValueError("Unknown cache split")\n    expected_y = metadata.label.map(LABELS).fillna(-1).to_numpy(dtype=np.int8)\n    if not np.array_equal(expected_y, arrays["y"]) or not (metadata.loc[metadata.split == "test", "y"] == -1).all():\n        raise ValueError("Invalid label encoding")\n    if not metadata.loc[metadata.split == "train", "y"].isin([0, 1]).all():\n        raise ValueError("Unlabeled training rows")\n    test_order = metadata.loc[metadata.split == "test", "test_order"].to_numpy()\n    if not np.array_equal(test_order, np.arange(len(test_order))):\n        raise ValueError("Invalid test ordering")\n    return arrays, metadata\n', 'eeg_comp/neural.py': '"""Small independently implemented end-to-end EEG classifier.\n\nInput is raw, named-channel EEG with its time axis intact. Normalization is\ncomputed independently for each trial and channel at each forward pass.\nThere is no fitted preprocessing or target-batch state.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nimport random\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass NeuralConfig:\n    channels: int = 8\n    temporal_filters: int = 8\n    spatial_multiplier: int = 2\n    temporal_kernel: int = 63\n    separable_kernel: int = 15\n    phase_bins: int = 8\n    dropout: float = 0.35\n    normalization_eps: float = 1e-5\n\n    def __post_init__(self) -> None:\n        for name in ("channels", "temporal_filters", "spatial_multiplier", "phase_bins"):\n            if getattr(self, name) < 1:\n                raise ValueError(f"{name} must be positive")\n        if any(value < 1 or value % 2 != 1 for value in (self.temporal_kernel, self.separable_kernel)):\n            raise ValueError("Convolution kernels must be positive and odd")\n        if not 0 <= self.dropout < 1 or self.normalization_eps <= 0:\n            raise ValueError("Invalid dropout or normalization epsilon")\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n\n\nclass TrialChannelStandardize(nn.Module):\n    """Normalize without mixing people, trials, or time windows."""\n\n    def __init__(self, eps: float = 1e-5):\n        super().__init__()\n        self.eps = eps\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        centered = x - x.mean(dim=-1, keepdim=True)\n        # Clamp before sqrt: sqrt(0) followed by a clamp has a NaN derivative\n        # on flat channels, even though its forward output is finite.\n        scale = centered.square().mean(dim=-1, keepdim=True).clamp_min(self.eps**2).sqrt()\n        return centered / scale\n\n\nclass PhaseConvNet(nn.Module):\n    """Temporal/spatial convolutions with phase-sensitive learned summaries.\n\nEight ordered temporal bins preserve coarse cue timing. Both the mean and\nstandard deviation of the *learned* representations enter the classifier;\nno handcrafted band powers, covariance matrices, or external features enter.\nGroup normalization has no running or cross-trial statistics.\n"""\n\n    def __init__(self, config: NeuralConfig | None = None):\n        super().__init__()\n        self.config = config or NeuralConfig()\n        cfg = self.config\n        width = cfg.temporal_filters * cfg.spatial_multiplier\n        self.standardize = TrialChannelStandardize(cfg.normalization_eps)\n        self.temporal = nn.Conv2d(\n            1, cfg.temporal_filters, (1, cfg.temporal_kernel),\n            padding=(0, cfg.temporal_kernel // 2), bias=False,\n        )\n        self.spatial = nn.Conv2d(\n            cfg.temporal_filters, width, (cfg.channels, 1),\n            groups=cfg.temporal_filters, bias=False,\n        )\n        self.norm1 = nn.GroupNorm(cfg.temporal_filters, width)\n        self.depthwise = nn.Conv2d(\n            width, width, (1, cfg.separable_kernel),\n            padding=(0, cfg.separable_kernel // 2), groups=width, bias=False,\n        )\n        self.pointwise = nn.Conv2d(width, width, 1, bias=False)\n        self.norm2 = nn.GroupNorm(cfg.temporal_filters, width)\n        self.dropout = nn.Dropout(cfg.dropout)\n        self.classifier = nn.Linear(width * cfg.phase_bins * 2, 1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.ndim != 3 or x.shape[1] != self.config.channels:\n            raise ValueError(f"Expected (trials,{self.config.channels},samples), got {tuple(x.shape)}")\n        if x.shape[-1] < 16 * self.config.phase_bins:\n            raise ValueError("Window too short for two pooling layers and the phase bins")\n        x = self.standardize(x).unsqueeze(1)\n        x = self.dropout(F.avg_pool2d(F.elu(self.norm1(self.spatial(self.temporal(x)))), (1, 4)))\n        x = self.dropout(F.avg_pool2d(F.elu(self.norm2(self.pointwise(self.depthwise(x)))), (1, 4)))\n        x = x.squeeze(2)\n        bins = torch.tensor_split(x, self.config.phase_bins, dim=-1)\n        means = torch.stack([part.mean(dim=-1) for part in bins], dim=-1)\n        second = torch.stack([part.square().mean(dim=-1) for part in bins], dim=-1)\n        deviations = (second - means.square()).clamp_min(1e-6).sqrt()\n        return self.classifier(torch.cat([means, deviations], dim=1).flatten(1)).squeeze(1)\n\n    def configuration(self) -> dict:\n        return asdict(self.config)\n\n\ndef binary_metrics(y: np.ndarray, p: np.ndarray) -> dict:\n    """Metrics only; this function never fits a predictor or threshold."""\n    y = np.asarray(y, dtype=np.int64)\n    p = np.asarray(p, dtype=np.float64)\n    if y.ndim != 1 or p.shape != y.shape:\n        raise ValueError("Expected matching one-dimensional labels and probabilities")\n    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Invalid binary labels or probabilities")\n    if len(y) == 0:\n        return {"n": 0}\n    pred = p >= 0.5\n    clipped = p.clip(1e-7, 1 - 1e-7)\n    counts = [int((y == label).sum()) for label in (0, 1)]\n    recalls = [float((pred[y == label] == label).mean()) for label in (0, 1) if counts[label]]\n    # Tied-score AUC, calculated without model-fitting dependencies.\n    auc = None\n    if all(counts):\n        _, inverse = np.unique(p, return_inverse=True)\n        pos = np.bincount(inverse, weights=y)\n        neg = np.bincount(inverse, weights=1 - y)\n        lower_neg = np.cumsum(neg) - neg\n        auc = float(np.sum(pos * (lower_neg + 0.5 * neg)) / (counts[0] * counts[1]))\n    return {\n        "n": int(len(y)), "rest_n": counts[0], "move_n": counts[1],\n        "accuracy": float(np.mean(pred == y)), "balanced_accuracy": float(np.mean(recalls)),\n        "log_loss": float(np.mean(-y * np.log(clipped) - (1-y) * np.log1p(-clipped))),\n        "brier": float(np.mean((p-y)**2)), "auc": auc,\n    }\n', 'eeg_comp/submission.py': '"""Validate the official zero-based ID contract before writing a candidate."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nEXPECTED_ROWS = {\'within_subject\':680,\'cross_subject\':360}\n\n\ndef build_submission(predictions, metadata, competition):\n    expected = metadata.loc[metadata.split.eq(\'test\')].sort_values([\'subject\',\'run\',\'trial_index\']).copy()\n    count = EXPECTED_ROWS[competition]\n    if len(expected) != count or not expected.competition.eq(competition).all():\n        raise ValueError(\'Wrong competition or test row count\')\n    if not np.array_equal(expected.test_order,np.arange(count)):\n        raise ValueError(\'Test order differs from participant/session/epoch ordering\')\n    if predictions.epoch_index.duplicated().any() or len(predictions) != count:\n        raise ValueError(\'Duplicate or missing prediction rows\')\n    aligned = predictions.set_index(\'epoch_index\').reindex(expected.epoch_index)\n    for column in [\'competition\',\'subject\',\'run\',\'file\',\'trial_index\',\'test_order\']:\n        if column not in aligned or not np.array_equal(aligned[column].to_numpy(),expected[column].to_numpy()):\n            raise ValueError(f\'Prediction mapping mismatch: {column}\')\n    probs = aligned.p_move.to_numpy(dtype=float)\n    if not np.isfinite(probs).all() or ((probs<0)|(probs>1)).any():\n        raise ValueError(\'Invalid or missing probabilities\')\n    return pd.DataFrame({\'ID\':np.arange(count),\'TARGET\':np.where(probs>=.5,\'move\',\'rest\')})\n\n\ndef save_candidate(prediction_path, metadata_path, competition, category, output):\n    prediction_path,metadata_path,output=map(Path,[prediction_path,metadata_path,output])\n    if output.exists():\n        raise FileExistsError(output)\n    frame=build_submission(pd.read_csv(prediction_path),pd.read_csv(metadata_path),competition)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    frame.to_csv(output,index=False)\n    manifest={\'competition\':competition,\'category\':category,\'rows\':len(frame),\n              \'id_min\':0,\'id_max\':len(frame)-1,\'threshold\':.5,\n              \'label_counts\':frame.TARGET.value_counts().to_dict(),\n              \'prediction_file\':str(prediction_path),\'prediction_sha256\':hashlib.sha256(prediction_path.read_bytes()).hexdigest(),\n              \'metadata_sha256\':hashlib.sha256(metadata_path.read_bytes()).hexdigest(),\n              \'csv_sha256\':hashlib.sha256(output.read_bytes()).hexdigest(),\n              \'status\':\'local candidate; not uploaded; Kaggle execution still required\',\n              \'ordering_source\':\'Official competition evaluation pages; see docs/RULES_AND_SUBMISSION.md\'}\n    output.with_suffix(\'.manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    return manifest\n', 'scripts/run_neural.py': '#!/usr/bin/env python3\n"""Fixed neural experiments with competition-specific leakage barriers.\n\nExample (allocated GPU):\n python scripts/run_neural.py --competition cross_subject --window task0_2 \\\n   --folds 0,1 --output results/cross_subject/neural_task0_2_pilot\nCPU execution is deliberately explicit and intended for small smoke tests.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport warnings\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom sklearn.model_selection import StratifiedKFold\n\nfrom eeg_comp.neural import NeuralConfig, PhaseConvNet, binary_metrics, seed_everything\n\n\nWINDOWS = {"task0_2": (0.0, 2.0), "task0_48": (0.0, 4.8), "baseline_m2_0": (-2.0, 0.0)}\nCHANNELS = ["Fz", "C3", "Cz", "C4", "PO7", "Pz", "PO8", "Oz"]\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef implementation_provenance() -> dict:\n    root = Path(__file__).resolve().parents[1]\n    return {name: file_sha256(root / name)\n            for name in ("eeg_comp/neural.py", "scripts/run_neural.py")}\n\n\ndef runtime_provenance() -> dict:\n    """Record numerical settings used by both training and checkpoint replay."""\n    return {"deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n            "cudnn_deterministic": torch.backends.cudnn.deterministic,\n            "cudnn_benchmark": torch.backends.cudnn.benchmark,\n            "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,\n            "cuda_matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,\n            "float32_matmul_precision": torch.get_float32_matmul_precision(),\n            "cudnn_version": torch.backends.cudnn.version(),\n            "cuda_version": torch.version.cuda,\n            "cpu_threads": torch.get_num_threads(),\n            "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG")}\n\n\n@dataclass\nclass Fold:\n    name: str\n    train: np.ndarray\n    inner_train: np.ndarray\n    inner_valid: np.ndarray\n    query: np.ndarray\n    seed: int\n    subject: str | None = None\n    confirmation: bool = False\n\n\ndef subject_seed(subject: str, seed: int) -> int:\n    """Person\'s seed does not depend on another supplied person\'s data/order."""\n    return (seed + int(hashlib.sha256(subject.encode()).hexdigest()[:8], 16)) % (2**31)\n\n\ndef stratified_halves(indices: np.ndarray, labels: np.ndarray, seed: int) -> list[np.ndarray]:\n    rng = np.random.default_rng(seed)\n    halves: list[list[int]] = [[], []]\n    for label in (0, 1):\n        selected = np.asarray(indices)[labels[indices] == label].copy()\n        if len(selected) < 2:\n            raise ValueError(f"Two-fold stratification needs two examples of each class, got {len(selected)} for {label}")\n        rng.shuffle(selected)\n        for half in (0, 1):\n            halves[half].extend(selected[half::2].tolist())\n    return [np.asarray(sorted(part), dtype=int) for part in halves]\n\n\ndef make_folds(meta: pd.DataFrame, labels: np.ndarray, competition: str,\n               seed: int, fold_ids: list[int] | None = None,\n               subjects: list[str] | None = None) -> list[Fold]:\n    if competition not in {"cross_subject", "within_subject"}:\n        raise ValueError(f"Unknown competition: {competition}")\n    is_train = meta["split"].eq("train").to_numpy()\n    if len(labels) != len(meta) or not np.isin(labels[is_train], [0, 1]).all():\n        raise ValueError("Labeled training rows must have binary labels")\n    people = meta["subject"].to_numpy()\n    run = meta["run"].to_numpy(dtype=int)\n    train_people = sorted(np.unique(people[is_train]))\n    folds = []\n    if competition == "cross_subject":\n        if subjects is not None:\n            raise ValueError("--subjects is available only for within_subject; Cross always uses the canonical full source cohort")\n        groups = np.array_split(np.asarray(train_people), 6)\n        confirmation_people = groups[-1]\n        for fold_id, group in enumerate(groups):\n            if fold_ids is not None and fold_id not in fold_ids:\n                continue\n            # A confirmation cohort must be absent from development fitting as\n            # well as scoring; otherwise outer-loop model choices can consume it.\n            excluded = group if fold_id == 5 else np.union1d(group, confirmation_people)\n            source_people = np.asarray([person for person in train_people if person not in excluded])\n            if len(group) == 0 or len(source_people) < 3:\n                raise ValueError("Cross grouped validation requires enough source participants")\n            # This split is fixed from IDs and seed, before reading their labels.\n            rng = np.random.default_rng(seed + fold_id)\n            inner_people = rng.permutation(source_people)[:min(3, len(source_people)-1)]\n            outer_train = np.flatnonzero(is_train & np.isin(people, source_people))\n            inner_valid = np.flatnonzero(is_train & np.isin(people, inner_people))\n            inner_train = np.setdiff1d(outer_train, inner_valid)\n            query = np.flatnonzero(is_train & np.isin(people, group))\n            folds.append(Fold(f"group_{fold_id}", outer_train, inner_train, inner_valid,\n                              query, seed + 1000 * fold_id, confirmation=fold_id == len(groups)-1))\n    else:\n        if subjects is not None and set(subjects) - set(train_people):\n            raise ValueError(f"Unknown source people: {sorted(set(subjects)-set(train_people))}")\n        for person in train_people:\n            if subjects is not None and person not in subjects:\n                continue\n            person_seed = subject_seed(person, seed)\n            calibration = np.flatnonzero(is_train & (people == person) & (run != 3))\n            feedback = np.flatnonzero(is_train & (people == person) & (run == 3))\n            if any(np.sum(labels[feedback] == label) < 2 for label in (0, 1)):\n                raise ValueError(f"{person} needs two run-3 labels of each class")\n            # Same precommitted split as the independent classical experiments;\n            # splitting is performed separately on this person\'s labels only.\n            splitter = StratifiedKFold(2, shuffle=True, random_state=seed)\n            halves = [feedback[query] for _, query in splitter.split(feedback, labels[feedback])]\n            calibration_runs = sorted(np.unique(run[calibration]))\n            if len(calibration_runs) >= 2:\n                inner_valid = calibration[run[calibration] == calibration_runs[-1]]\n                inner_calibration = np.setdiff1d(calibration, inner_valid)\n            else:\n                inner_calibration, inner_valid = stratified_halves(calibration, labels, person_seed + 31)\n            for half, query in enumerate(halves):\n                if fold_ids is not None and half not in fold_ids:\n                    continue\n                support = halves[1-half]\n                train = np.sort(np.concatenate([calibration, support]))\n                inner_train = np.sort(np.concatenate([inner_calibration, support]))\n                folds.append(Fold(f"{person}_half_{half}", train, inner_train, inner_valid,\n                                  query, person_seed + half, subject=person))\n    if not folds:\n        raise ValueError("No folds requested")\n    for fold in folds:\n        assert len(fold.query) and len(fold.train) and len(fold.inner_train) and len(fold.inner_valid)\n        assert not np.intersect1d(fold.query, fold.train).size\n        assert not np.intersect1d(fold.inner_train, fold.inner_valid).size\n        assert np.array_equal(np.union1d(fold.inner_train, fold.inner_valid), fold.train)\n        if competition == "cross_subject":\n            assert not set(people[fold.query]) & set(people[fold.train])\n            assert not set(people[fold.inner_train]) & set(people[fold.inner_valid])\n        else:\n            assert set(people[np.concatenate([fold.train, fold.query])]) == {fold.subject}\n    return folds\n\n\ndef load_cache(cache_dir: Path, competition: str, window: str) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, dict]:\n    metadata_path = cache_dir / "metadata.csv"\n    meta = pd.read_csv(metadata_path, keep_default_na=False)\n    required = {"epoch_index", "competition", "split", "subject", "run", "label", "test_order"}\n    if not required.issubset(meta.columns):\n        raise ValueError(f"Missing metadata columns: {sorted(required-set(meta.columns))}")\n    if set(meta["competition"]) != {competition}:\n        raise ValueError("Cache competition differs from the requested competition")\n    if set(meta["split"]) != {"train", "test"} or not meta["run"].isin([1, 2, 3]).all():\n        raise ValueError("Expected train/test rows and run IDs 1,2,3")\n    if not np.array_equal(meta["epoch_index"], np.arange(len(meta))):\n        raise ValueError("Metadata row order does not match epoch_index")\n    with np.load(cache_dir / "epochs.npz", allow_pickle=False) as cache:\n        fs = int(cache["fs"])\n        cue = int(cache["cue_index"])\n        if fs != 250 or cue != 750 or cache["channels"].tolist() != CHANNELS:\n            raise ValueError("Unexpected sample rate, cue index, or named EEG channel order")\n        start, end = [cue + round(value * fs) for value in WINDOWS[window]]\n        raw = cache["X"]\n        if raw.shape != (len(meta), 8, 2000) or start < 0 or end > raw.shape[-1]:\n            raise ValueError(f"Unexpected epoch shape/window: {raw.shape}, [{start},{end})")\n        x = np.ascontiguousarray(raw[:, :, start:end], dtype=np.float32)\n        masks = [cache[key] for key in ("valid_mask", "phase_valid_mask")]\n        if any(mask.shape != (len(meta), 2000) or mask.dtype != np.bool_ for mask in masks):\n            raise ValueError("Expected boolean native-sample validity masks")\n        valid = masks[0][:, start:end] & masks[1][:, start:end]\n        bad = np.flatnonzero(~valid.all(axis=1) | ~np.isfinite(x).all(axis=(1, 2)))\n        if len(bad):\n            raise ValueError(f"{len(bad)} trials invalid in {window}; no automatic padding/exclusion. First epoch indices: {bad[:20].tolist()}")\n        labels = meta["label"].map({"rest": 0, "move": 1, "": -1}).to_numpy()\n        if pd.isna(labels).any():\n            raise ValueError("Unexpected label mapping")\n        labels = labels.astype(np.int64)\n        if cache["y"].dtype.kind not in "iu" or not np.array_equal(cache["y"], labels):\n            raise ValueError("Cache labels differ from metadata")\n    if not np.array_equal(labels >= 0, meta["split"].eq("train")):\n        raise ValueError("Train/test label visibility mismatch")\n    if not np.array_equal(meta.loc[meta["split"].eq("test"), "test_order"], np.arange((labels < 0).sum())):\n        raise ValueError("Test ordering must be complete and contiguous")\n    cache_info = {"cache_directory": str(cache_dir.resolve()), "fs": fs, "cue_index": cue,\n                  "crop_samples": [start, end], "channels": CHANNELS,\n                  "metadata_sha256": hashlib.sha256(metadata_path.read_bytes()).hexdigest(),\n                  "epochs_sha256": file_sha256(cache_dir / "epochs.npz"),\n                  "audit_sha256": file_sha256(cache_dir / "audit.json") if (cache_dir / "audit.json").exists() else None,\n                  "normalization": "runtime per trial per channel; no fitted state",\n                  "filtering": "none", "invalid_crop_policy": "fail; no padding or trial removal"}\n    return x, labels, meta, cache_info\n\n\ndef protocol_weights(meta: pd.DataFrame) -> dict[int, float]:\n    counts = meta.loc[meta["split"].eq("test"), "run"].value_counts()\n    if not len(counts):\n        raise ValueError("Need test run inventory to report the target protocol mixture")\n    return {int(run): float(count / counts.sum()) for run, count in counts.items()}\n\n\ndef weighted_protocol_loss(y: np.ndarray, p: np.ndarray, runs: np.ndarray, weights: dict[int, float]) -> float:\n    available = {run: weight for run, weight in weights.items() if np.any(runs == run)}\n    if not available:\n        # Within-person stopping uses calibration runs, while target is run 3.\n        return binary_metrics(y, p)["log_loss"]\n    return sum(weight * binary_metrics(y[runs == run], p[runs == run])["log_loss"]\n               for run, weight in available.items()) / sum(available.values())\n\n\ndef report_metrics(frame: pd.DataFrame, weights: dict[int, float]) -> dict:\n    y = frame["y"].to_numpy()\n    p = frame["p_move"].to_numpy()\n    result = binary_metrics(y, p)\n    result["per_run"] = {str(run): binary_metrics(group["y"], group["p_move"])\n                         for run, group in frame.groupby("run")}\n    result["per_subject"] = {str(person): binary_metrics(group["y"], group["p_move"])\n                             for person, group in frame.groupby("subject")}\n    available = {run: weight for run, weight in weights.items() if str(run) in result["per_run"]}\n    result["target_run_weights"] = weights\n    result["target_weighted_accuracy"] = sum(\n        weight * result["per_run"][str(run)]["accuracy"] for run, weight in available.items()\n    ) / sum(available.values()) if available else None\n    result["missing_target_runs"] = sorted(set(weights) - set(available))\n    return result\n\n\ndef predict(model: PhaseConvNet, x: np.ndarray, indices: np.ndarray, device: torch.device,\n            batch_size: int) -> tuple[np.ndarray, np.ndarray]:\n    if not len(indices):\n        raise ValueError("No trials selected for inference")\n    model.eval()\n    values = []\n    with torch.no_grad():\n        for start in range(0, len(indices), batch_size):\n            batch = torch.from_numpy(x[indices[start:start+batch_size]]).to(device)\n            values.append(model(batch).cpu().numpy())\n    logits = np.concatenate(values)\n    # Float64 sigmoid remains stable even if training produced large logits.\n    probabilities = np.exp(-np.logaddexp(0, -logits.astype(np.float64)))\n    if not np.isfinite(probabilities).all():\n        raise RuntimeError("Model produced nonfinite predictions")\n    return logits, probabilities\n\n\ndef fit(x: np.ndarray, y: np.ndarray, meta: pd.DataFrame, train: np.ndarray,\n        valid: np.ndarray | None, *, seed: int, config: NeuralConfig,\n        args: argparse.Namespace, device: torch.device, epochs: int,\n        weights: dict[int, float], history_path: Path) -> tuple[PhaseConvNet, int]:\n    if set(np.unique(y[train])) != {0, 1}:\n        raise ValueError("Training fold lacks one class")\n    if valid is not None and (np.intersect1d(train, valid).size or not np.isin(y[valid], [0, 1]).all()):\n        raise ValueError("Inner validation must use separate labeled trials")\n    seed_everything(seed)\n    model = PhaseConvNet(config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)\n    loss_weights = np.ones(len(train), dtype=np.float32)\n    if args.protocol_balanced_loss:\n        strata = meta.iloc[train]["subject"].astype(str) + "/" + meta.iloc[train]["run"].astype(str)\n        counts = strata.value_counts()\n        loss_weights = (1 / strata.map(counts)).to_numpy(dtype=np.float32)\n        loss_weights /= loss_weights.mean()\n    dataset = TensorDataset(torch.from_numpy(x[train]), torch.from_numpy(y[train]).float(),\n                            torch.from_numpy(loss_weights))\n    generator = torch.Generator().manual_seed(seed)\n    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=0,\n                        generator=generator, pin_memory=device.type == "cuda")\n    best_loss, best_epoch, stale = float("inf"), epochs, 0\n    best_state = None\n    with history_path.open("w") as history:\n        for epoch in range(1, epochs+1):\n            model.train()\n            total_loss = 0.0\n            for batch, targets, sample_weights in loader:\n                batch, targets, sample_weights = [value.to(device) for value in (batch, targets, sample_weights)]\n                optimizer.zero_grad(set_to_none=True)\n                losses = F.binary_cross_entropy_with_logits(model(batch), targets, reduction="none")\n                loss = (losses * sample_weights).mean()\n                if not torch.isfinite(loss):\n                    raise RuntimeError("Nonfinite training loss")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                optimizer.step()\n                total_loss += float(loss.detach()) * len(batch)\n            record = {"epoch": epoch, "train_loss": total_loss / len(train)}\n            if valid is not None:\n                _, prob = predict(model, x, valid, device, args.batch_size)\n                val_loss = weighted_protocol_loss(y[valid], prob, meta.iloc[valid]["run"].to_numpy(), weights)\n                record.update(valid_loss=val_loss, valid_accuracy=float(np.mean((prob >= 0.5) == y[valid])))\n                if val_loss < best_loss - 1e-4:\n                    best_loss, best_epoch, stale = val_loss, epoch, 0\n                    best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n                else:\n                    stale += 1\n            history.write(json.dumps(record) + "\\n")\n            history.flush()\n            if epoch == 1 or epoch % 10 == 0:\n                print(json.dumps({"training": history_path.stem, **record}), flush=True)\n            if valid is not None and stale >= args.patience:\n                break\n    if valid is not None:\n        if best_state is None:\n            raise RuntimeError("No finite validation checkpoint")\n        model.load_state_dict(best_state)\n    return model, best_epoch\n\n\ndef checkpoint(path: Path, model: PhaseConvNet, args: argparse.Namespace, *,\n               epochs: int, seed: int, train: np.ndarray, cache_info: dict) -> None:\n    torch.save({"model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "model_config": model.configuration(), "competition": args.competition,\n                "window": args.window, "epochs": epochs, "seed": seed,\n                "train_epoch_indices": train.tolist(), "cache_info": cache_info,\n                "implementation_sha256": implementation_provenance(),\n                "runtime": runtime_provenance(),\n                "label_mapping": {"rest": 0, "move": 1}}, path)\n\n\ndef save_test_predictions(frames: list[pd.DataFrame], output: Path) -> None:\n    combined = pd.concat(frames).sort_values("test_order")\n    if not np.array_equal(combined["test_order"], np.arange(len(combined))):\n        raise ValueError("Test order is incomplete or duplicated")\n    combined.to_csv(output / "test_predictions.csv", index=False)\n    print(json.dumps({"saved_test_predictions": len(combined),\n                      "path": str(output / "test_predictions.csv")}), flush=True)\n\n\ndef inference_from_checkpoints(args: argparse.Namespace, x: np.ndarray, meta: pd.DataFrame,\n                               cache_info: dict, device: torch.device) -> None:\n    """Reproduce final predictions using saved person-specific/source models."""\n    people = [None] if args.competition == "cross_subject" else sorted(meta.loc[meta["split"].eq("test"), "subject"].unique())\n    frames = []\n    for person in people:\n        state = torch.load(args.inference_from / "final" / f"{person or \'all_sources\'}.pt",\n                           map_location="cpu", weights_only=True)\n        if (state["competition"] != args.competition or state["window"] != args.window\n                or state["label_mapping"] != {"rest": 0, "move": 1}):\n            raise ValueError("Checkpoint competition, window, or label mapping mismatch")\n        for key in ("metadata_sha256", "crop_samples", "fs", "cue_index", "channels"):\n            if state["cache_info"].get(key) != cache_info[key]:\n                raise ValueError(f"Checkpoint cache contract mismatch: {key}")\n        if "epochs_sha256" in state["cache_info"]:\n            if state["cache_info"]["epochs_sha256"] != cache_info["epochs_sha256"]:\n                raise ValueError("Checkpoint cache contract mismatch: epochs_sha256")\n        else:\n            warnings.warn("Legacy checkpoint has no signal-cache hash; historical raw-cache identity is unverified.")\n        saved_implementation = state.get("implementation_sha256", {})\n        if saved_implementation.get("eeg_comp/neural.py") not in (None, implementation_provenance()["eeg_comp/neural.py"]):\n            raise ValueError("Checkpoint neural implementation hash differs from current source")\n        trained = meta.iloc[state["train_epoch_indices"]]\n        expected = meta.loc[meta["split"].eq("train") & (True if person is None else meta["subject"].eq(person))]\n        if not np.array_equal(trained["epoch_index"], expected["epoch_index"]):\n            raise ValueError("Final checkpoint training indices violate the competition/person boundary")\n        model = PhaseConvNet(NeuralConfig(**state["model_config"]))\n        model.load_state_dict(state["model_state_dict"], strict=True)\n        model.to(device)\n        selected = meta["split"].eq("test") & (True if person is None else meta["subject"].eq(person))\n        indices = np.flatnonzero(selected)\n        logits, probability = predict(model, x, indices, device, args.batch_size)\n        frame = meta.iloc[indices].copy()\n        frame["logit_move"], frame["p_move"] = logits, probability\n        frame["prediction"] = np.where(probability >= 0.5, "move", "rest")\n        frames.append(frame)\n    save_test_predictions(frames, args.output)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--competition", required=True, choices=["within_subject", "cross_subject"])\n    parser.add_argument("--cache", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--window", choices=WINDOWS, default="task0_2")\n    parser.add_argument("--folds", help="Comma-separated canonical outer fold IDs (Cross 0..5; Within 0,1)")\n    parser.add_argument("--subjects", help="Within-only person subset, e.g. S001,S002; useful for smoke tests")\n    parser.add_argument("--seed", type=int, default=20261003)\n    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda")\n    parser.add_argument("--threads", type=int, default=4)\n    parser.add_argument("--disable-tf32", action="store_true",\n                        help="Explicit numerical contrast: disable CUDA matmul and cuDNN TF32")\n    parser.add_argument("--max-epochs", type=int, default=60)\n    parser.add_argument("--patience", type=int, default=10)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--learning-rate", type=float, default=0.001)\n    parser.add_argument("--weight-decay", type=float, default=0.01)\n    parser.add_argument("--protocol-balanced-loss", action="store_true",\n                        help="Explicit controlled contrast: equal-weight subject/run strata in training")\n    parser.add_argument("--predict-test", action="store_true", help="After full validation, refit final models and save test probabilities")\n    parser.add_argument("--inference-from", type=Path,\n                        help="Reload final/*.pt from a completed experiment; no fitting or validation")\n    args = parser.parse_args()\n    if min(args.max_epochs, args.patience, args.batch_size, args.threads) < 1:\n        parser.error("Epochs, patience, batch size, and threads must be positive")\n    args.cache = args.cache or Path("artifacts") / args.competition\n    args.folds = [int(value) for value in args.folds.split(",")] if args.folds else None\n    args.subjects = args.subjects.split(",") if args.subjects else None\n    allowed = set(range(6 if args.competition == "cross_subject" else 2))\n    if args.folds is not None and (len(args.folds) != len(set(args.folds)) or set(args.folds) - allowed):\n        parser.error(f"Unique fold IDs required from {sorted(allowed)}")\n    if args.predict_test and (args.subjects is not None or (args.folds is not None and set(args.folds) != allowed)):\n        parser.error("--predict-test requires full validation of all folds and people")\n    if args.inference_from and (args.predict_test or args.subjects is not None or args.folds is not None):\n        parser.error("--inference-from cannot be combined with --predict-test, --subjects, or --folds")\n    if args.folds is None:\n        args.folds = sorted(allowed) if args.predict_test else list(range(5 if args.competition == "cross_subject" else 2))\n    return args\n\n\ndef main() -> None:\n    args = parse_args()\n    torch.set_num_threads(args.threads)\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("No allocated CUDA GPU. Use PBS GPU resources; --device cpu is for explicit small smoke tests.")\n    # Training calls this inside fit(); checkpoint-only inference needs the same\n    # deterministic backend policy before its first convolution as well.\n    seed_everything(args.seed)\n    if args.disable_tf32:\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n    device = torch.device(args.device)\n    x, y, meta, cache_info = load_cache(args.cache, args.competition, args.window)\n    weights = protocol_weights(meta)\n    folds = [] if args.inference_from else make_folds(meta, y, args.competition, args.seed, args.folds, args.subjects)\n    if (args.output / "config.json").exists():\n        raise FileExistsError("Output already has config.json; choose a new experiment directory")\n    args.output.mkdir(parents=True, exist_ok=True)\n    config = NeuralConfig()\n    config_record = {"arguments": {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()},\n                     "model": asdict(config), "cache": cache_info, "torch_version": str(torch.__version__),\n                     "implementation_sha256": implementation_provenance(),\n                     "runtime": runtime_provenance(),\n                     "device": str(device), "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,\n                     "category": "end-to-end DL", "test_run_weights": weights,\n                     "epoch_selection": "inner source-person validation for Cross; own-person calibration validation for Within",\n                     "experiment_status": "fixed researcher-designed baseline; no automatic hyperparameter search",\n                     "inference_mode": "saved checkpoint" if args.inference_from else "training",\n                     "confirmation_policy": "group5 absent from fitting and scoring of development groups0..4",\n                     "confirmation_group": 5 if args.competition == "cross_subject" else None}\n    (args.output / "config.json").write_text(json.dumps(config_record, indent=2) + "\\n")\n    if args.inference_from:\n        inference_from_checkpoints(args, x, meta, cache_info, device)\n        return\n    all_oof = []\n    fold_records = []\n    experiment_start = time.monotonic()\n    for fold in folds:\n        fold_dir = args.output / fold.name\n        fold_dir.mkdir(exist_ok=True)\n        indices = {key: getattr(fold, key).tolist() for key in ("train", "inner_train", "inner_valid", "query")}\n        (fold_dir / "split.json").write_text(json.dumps({**indices, "subject": fold.subject,\n            "seed": fold.seed, "confirmation": fold.confirmation}, indent=2) + "\\n")\n        print(json.dumps({"fold": fold.name, "train_n": len(fold.train), "query_n": len(fold.query),\n                          "query_people": sorted(meta.iloc[fold.query]["subject"].unique().tolist())}), flush=True)\n        _, best_epoch = fit(x, y, meta, fold.inner_train, fold.inner_valid, seed=fold.seed,\n            config=config, args=args, device=device, epochs=args.max_epochs, weights=weights,\n            history_path=fold_dir / "inner_history.jsonl")\n        # Reinitialize and refit all source data, without consulting outer query labels.\n        model, _ = fit(x, y, meta, fold.train, None, seed=fold.seed, config=config, args=args,\n            device=device, epochs=best_epoch, weights=weights, history_path=fold_dir / "refit_history.jsonl")\n        logits, prob = predict(model, x, fold.query, device, args.batch_size)\n        frame = meta.iloc[fold.query].copy()\n        frame["y"], frame["logit_move"], frame["p_move"] = y[fold.query], logits, prob\n        frame["prediction"] = np.where(prob >= 0.5, "move", "rest")\n        frame["fold"], frame["confirmation"] = fold.name, fold.confirmation\n        frame.to_csv(fold_dir / "oof.csv", index=False)\n        checkpoint(fold_dir / "model.pt", model, args, epochs=best_epoch, seed=fold.seed,\n                   train=fold.train, cache_info=cache_info)\n        score = report_metrics(frame, weights)\n        record = {"fold": fold.name, "subject": fold.subject, "selected_epochs": best_epoch,\n                  "confirmation": fold.confirmation, "metrics": score}\n        fold_records.append(record)\n        all_oof.append(frame)\n        combined = pd.concat(all_oof).sort_values("epoch_index")\n        if combined["epoch_index"].duplicated().any():\n            raise RuntimeError("Duplicated out-of-fold trials")\n        combined.to_csv(args.output / "oof.csv", index=False)\n        summary = {"metrics": report_metrics(combined, weights), "folds": fold_records,\n                   "elapsed_seconds": time.monotonic() - experiment_start,\n                   "complete_requested_validation": len(fold_records) == len(folds)}\n        development = combined.loc[~combined["confirmation"]]\n        confirmation = combined.loc[combined["confirmation"]]\n        if len(development):\n            summary["development_metrics"] = report_metrics(development, weights)\n        if len(confirmation):\n            summary["confirmation_metrics"] = report_metrics(confirmation, weights)\n        (args.output / "metrics.json").write_text(json.dumps(summary, indent=2) + "\\n")\n        print(json.dumps({"completed": fold.name, "epochs": best_epoch,\n                          "accuracy": score["accuracy"], "target_weighted_accuracy": score["target_weighted_accuracy"]}), flush=True)\n    if args.predict_test:\n        final_frames = []\n        final_dir = args.output / "final"\n        final_dir.mkdir(exist_ok=True)\n        people = [None] if args.competition == "cross_subject" else sorted(meta.loc[meta["split"].eq("test"), "subject"].unique())\n        for person in people:\n            person_rows = np.ones(len(meta), dtype=bool) if person is None else meta["subject"].eq(person).to_numpy()\n            train = np.flatnonzero(person_rows & meta["split"].eq("train").to_numpy())\n            test = np.flatnonzero(person_rows & meta["split"].eq("test").to_numpy())\n            chosen = [record["selected_epochs"] for record in fold_records if record["subject"] == person]\n            epochs = max(1, round(float(np.median(chosen))))\n            seed = args.seed if person is None else subject_seed(person, args.seed)\n            name = person or "all_sources"\n            model, _ = fit(x, y, meta, train, None, seed=seed, config=config, args=args,\n                device=device, epochs=epochs, weights=weights, history_path=final_dir / f"{name}_history.jsonl")\n            logits, prob = predict(model, x, test, device, args.batch_size)\n            frame = meta.iloc[test].copy()\n            frame["logit_move"], frame["p_move"] = logits, prob\n            frame["prediction"] = np.where(prob >= 0.5, "move", "rest")\n            final_frames.append(frame)\n            checkpoint(final_dir / f"{name}.pt", model, args, epochs=epochs, seed=seed, train=train, cache_info=cache_info)\n        save_test_predictions(final_frames, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'eeg_comp/__init__.py': '"""Embedded pure raw-EEG neural pipeline."""\n', 'scripts/__init__.py': '"""Embedded fitting implementation."""\n'}
source_root = OUTPUT_DIR / 'embedded_source'
source_root.mkdir(parents=True, exist_ok=True)
for filename, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[filename]:
        raise ValueError('Embedded source checksum mismatch: ' + filename)
    path = source_root / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
if any(name == 'eeg_comp' or name.startswith('eeg_comp.') or name == 'scripts' or name.startswith('scripts.') for name in sys.modules):
    raise RuntimeError('Restart the kernel before executing this standalone notebook')
sys.path.insert(0, str(source_root.resolve()))
importlib.invalidate_caches()
import numpy as np
import pandas as pd
import sklearn
import torch
from eeg_comp.data import CHANNELS, FS, CUE_INDEX, identify_file, parse_markers, extract_epochs
from eeg_comp.neural import NeuralConfig, PhaseConvNet, seed_everything
from eeg_comp.submission import build_submission
from scripts.run_neural import (WINDOWS, checkpoint, fit, make_folds, predict,
                                protocol_weights, runtime_provenance, subject_seed)
torch.set_num_threads(THREADS)
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Enable a CUDA GPU for the full-retraining notebook')
device = torch.device(DEVICE)
seed_everything(SEED)
# Explicitly preserve the original GPU experiment's numerical policy.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'torch': str(torch.__version__)}
print(json.dumps({'competition': COMPETITION, 'device': str(device), 'versions': VERSIONS,
                  'runtime': runtime_provenance()}, indent=2))


## Load this competition from raw recordings


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def locate_data_root(explicit_root):
    # Never recursively search /kaggle/input: the other challenge may be attached.
    if explicit_root is not None:
        candidates = [Path(explicit_root)]
    else:
        candidates = [Path('/kaggle/input') / SLUG,
                      Path('/kaggle/input/competitions') / SLUG]
    found = [root.resolve() for root in candidates
             if (root / 'Training set').is_dir() and (root / TEST_FOLDER).is_dir()]
    if len(set(found)) != 1:
        raise FileNotFoundError('Set DATA_ROOT to this challenge folder containing '
                                f'Training set and {TEST_FOLDER!r}; candidates={candidates}')
    root = found[0]
    # Explicit roots must still have the selected challenge's test layout only.
    other_test = ('Cross participant test set' if COMPETITION == 'within_subject'
                  else 'Single subject test set')
    if (root / other_test).exists():
        raise ValueError('Both competition test folders found in one input root')
    return root


def load_raw_competition(root):
    arrays, masks, phase_masks, rows, file_manifest = [], [], [], [], []
    test_order = 0
    for split, folder, expected_files in [('train', 'Training set', 50),
                                          ('test', TEST_FOLDER, EXPECTED_TEST_FILES)]:
        paths = list((root / folder).glob('*.csv'))
        paths.sort(key=lambda path: (*identify_file(path, COMPETITION, split), path.name))
        if len(paths) != expected_files:
            raise ValueError(f'{split}: expected {expected_files} EEG files, found {len(paths)}')
        for path in paths:
            subject, run = identify_file(path, COMPETITION, split)
            frame = pd.read_csv(path)
            trials, marker_report = parse_markers(frame['Marker_val'], split)
            X_part, valid, phase_valid, trial_rows, timing = extract_epochs(frame, trials)
            # Fail closed: no filling, padding, dropping, or substituting unsupported windows.
            for lo, hi in [(-2., 0.), (0., 4.8)]:
                a, b = CUE_INDEX + round(lo * FS), CUE_INDEX + round(hi * FS)
                if not (valid[:, a:b] & phase_valid[:, a:b]).all():
                    raise ValueError(f'{path.name}: unsupported required phase {lo, hi}')
            for row in trial_rows:
                row.update(epoch_index=len(rows), competition=COMPETITION, split=split,
                           subject=subject, run=run, file=path.relative_to(root).as_posix(),
                           test_order=test_order if split == 'test' else -1)
                if split == 'test':
                    test_order += 1
                rows.append(row)
            file_manifest.append({'file': path.relative_to(root).as_posix(), 'split': split,
                                  'subject': subject, 'run': run, 'trials': len(trials),
                                  'sha256': sha256_file(path), **marker_report,
                                  'median_dt_seconds': timing['median_dt_seconds']})
            arrays.append(X_part)
            masks.append(valid)
            phase_masks.append(phase_valid)
            print(f'{COMPETITION}: {path.name}, {len(trials)} trials', flush=True)
    meta = pd.DataFrame(rows)
    X, valid, phase_valid = map(np.concatenate, [arrays, masks, phase_masks])
    train, test = meta[meta.split.eq('train')], meta[meta.split.eq('test')]
    if len(train) != 1795 or len(test) != EXPECTED_TEST_ROWS:
        raise ValueError('Raw trial counts differ from audited competition release')
    if train.subject.nunique() != 17 or test.subject.nunique() != EXPECTED_TEST_PEOPLE:
        raise ValueError('Unexpected participant counts')
    if not set(train.y.unique()) == {0, 1} or not test.y.eq(-1).all():
        raise ValueError('Invalid label visibility')
    if COMPETITION == 'within_subject':
        if set(train.subject) != set(test.subject) or not test.run.eq(3).all():
            raise ValueError('Within-subject participant/run mismatch')
    elif set(train.subject) & set(test.subject):
        raise ValueError('Cross-subject targets overlap training people')
    ordered = test.sort_values(['subject', 'run', 'trial_index'])
    if not np.array_equal(ordered.test_order, np.arange(EXPECTED_TEST_ROWS)):
        raise ValueError('Test order is not participant/session/original epoch order')
    if ordered.duplicated(['file', 'trial_index']).any():
        raise ValueError('Duplicate test trial mapping')
    if not ordered.groupby('file').size().eq(40).all():
        raise ValueError('Expected exactly forty explicitly marked trials per test file')
    return X, valid, phase_valid, meta, file_manifest


data_root = locate_data_root(DATA_ROOT)
X, valid, phase_valid, meta, input_files = load_raw_competition(data_root)
y = meta.y.to_numpy(dtype=np.int8)
meta.to_csv(OUTPUT_DIR / 'metadata.csv', index=False)
print(meta.groupby(['split', 'run']).size())


## Train independent neural models and replay checkpoints


In [ ]:
# The network receives only raw EEG in [0,4.8), without filtering or features.
a, b = [CUE_INDEX + round(t * FS) for t in WINDOWS[WINDOW]]
if a < 0 or b > X.shape[-1] or not (valid[:, a:b] & phase_valid[:, a:b]).all():
    raise ValueError('Unsupported native EEG task crop')
x = np.ascontiguousarray(X[:, :, a:b], dtype=np.float32)
y = np.asarray(y, dtype=np.int64)
if not np.isfinite(x).all():
    raise ValueError('Nonfinite neural input')
if not np.array_equal(y >= 0, meta.split.eq('train')):
    raise ValueError('Training/test label visibility differs')
config = NeuralConfig()
args = argparse.Namespace(competition=COMPETITION, window=WINDOW,
                         batch_size=64, learning_rate=.001, weight_decay=.01,
                         protocol_balanced_loss=False, patience=10)
weights = protocol_weights(meta)
cache_info = {'raw_inputs': input_files, 'fs': FS, 'cue_index': CUE_INDEX,
              'crop_samples': [a, b], 'channels': list(CHANNELS),
              'metadata_sha256': sha256_file(OUTPUT_DIR / 'metadata.csv'),
              'normalization': 'runtime per trial per channel; no fitted state',
              'filtering': 'none', 'invalid_crop_policy': 'fail; no padding or trial removal'}
final_dir = OUTPUT_DIR / 'final'
final_dir.mkdir(exist_ok=True)
inner_dir = OUTPUT_DIR / 'inner'
inner_dir.mkdir(exist_ok=True)
epoch_records = []
if COMPETITION == 'within_subject':
    # Identical fold definitions and independent seeded stopping fits to run_neural.
    # Outer query labels are never scored or used to select any setting here.
    for fold in make_folds(meta, y, COMPETITION, SEED, fold_ids=[0, 1]):
        _, selected = fit(x, y, meta, fold.inner_train, fold.inner_valid,
                          seed=fold.seed, config=config, args=args, device=device,
                          epochs=MAX_EPOCHS, weights=weights,
                          history_path=inner_dir / (fold.name + '_history.jsonl'))
        epoch_records.append({'fold': fold.name, 'participant': fold.subject,
                              'seed': fold.seed, 'selected_epochs': int(selected),
                              'inner_train_epoch_indices': fold.inner_train.tolist(),
                              'inner_valid_epoch_indices': fold.inner_valid.tolist(),
                              'unused_outer_query_epoch_indices': fold.query.tolist()})
else:
    # Fixed from the five development fits' inner-source stopping epochs only.
    # The reserved confirmation people are included in final fitting, never scored.
    epoch_records = [{'selected_epochs': CROSS_EPOCHS,
                      'selection': 'fixed median of development inner-source stopping epochs',
                      'development_inner_epochs': [19, 13, 19, 13, 37],
                      'confirmation_labels_used_for_selection': False}]
(OUTPUT_DIR / 'epoch_selection.json').write_text(json.dumps(epoch_records, indent=2) + '\n')

people = sorted(meta.loc[meta.split.eq('test'), 'subject'].unique()) if COMPETITION == 'within_subject' else [None]
frames, fit_manifest, replay_records = [], [], []
for person in people:
    own = np.ones(len(meta), dtype=bool) if person is None else meta.subject.eq(person).to_numpy()
    train = np.flatnonzero(own & meta.split.eq('train').to_numpy())
    query = np.flatnonzero(own & meta.split.eq('test').to_numpy())
    if person is not None and set(meta.iloc[train].subject) != {person}:
        raise ValueError('Within-subject model crossed a participant boundary')
    chosen = [r['selected_epochs'] for r in epoch_records if r.get('participant') == person]
    epochs = max(1, round(float(np.median(chosen))))
    seed = SEED if person is None else subject_seed(person, SEED)
    name = person or 'all_sources'
    model, _ = fit(x, y, meta, train, None, seed=seed, config=config, args=args,
                   device=device, epochs=epochs, weights=weights,
                   history_path=final_dir / (name + '_history.jsonl'))
    logits, probability = predict(model, x, query, device, args.batch_size)
    checkpoint_path = final_dir / (name + '.pt')
    checkpoint(checkpoint_path, model, args, epochs=epochs, seed=seed,
               train=train, cache_info=cache_info)
    # Strict immediate replay on the same numerical backend; no tolerance widening.
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if (saved['competition'] != COMPETITION or saved['window'] != WINDOW
            or saved['label_mapping'] != {'rest': 0, 'move': 1}
            or saved['train_epoch_indices'] != train.tolist()
            or saved['cache_info'] != cache_info):
        raise ValueError('Saved neural checkpoint contract differs')
    replay = PhaseConvNet(NeuralConfig(**saved['model_config'])).to(device)
    replay.load_state_dict(saved['model_state_dict'], strict=True)
    replay_logits, replay_probability = predict(replay, x, query, device, args.batch_size)
    np.testing.assert_allclose(replay_probability, probability, rtol=0, atol=1e-6)
    np.testing.assert_allclose(replay_logits, logits, rtol=0, atol=1e-5)
    np.testing.assert_array_equal(replay_probability >= .5, probability >= .5)
    replay_records.append({'participant': person, 'rows': len(query),
                           'same_device': str(device), 'labels_identical': True,
                           'max_abs_probability_difference': float(np.max(np.abs(replay_probability - probability))),
                           'max_abs_logit_difference': float(np.max(np.abs(replay_logits - logits))),
                           'probability_atol': 1e-6, 'logit_atol': 1e-5})
    frame = meta.iloc[query].copy()
    frame['logit_move'], frame['p_move'] = logits, probability
    frame['prediction'] = np.where(probability >= .5, 'move', 'rest')
    frames.append(frame)
    fit_manifest.append({'participant': person, 'seed': int(seed), 'epochs': epochs,
                         'training_trials': len(train), 'test_trials': len(query),
                         'training_participants': sorted(meta.iloc[train].subject.unique()),
                         'training_epoch_indices': train.tolist(),
                         'test_epoch_indices': query.tolist(),
                         'checkpoint': checkpoint_path.relative_to(OUTPUT_DIR).as_posix(),
                         'checkpoint_sha256': sha256_file(checkpoint_path)})
    print(f'Fitted and replayed {name}: {len(train)} train, {len(query)} test, {epochs} epochs', flush=True)
test = pd.concat(frames).sort_values('test_order')
if not np.array_equal(test.test_order, np.arange(len(test))):
    raise ValueError('Incomplete or duplicated test prediction mapping')
test.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
(OUTPUT_DIR / 'checkpoint_replay.json').write_text(json.dumps(replay_records, indent=2) + '\n')


## Export and verify the submission


In [ ]:
submission = build_submission(test, meta, COMPETITION)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
reloaded = pd.read_csv(OUTPUT_DIR / 'submission.csv')
if list(reloaded.columns) != ['ID', 'TARGET'] or not np.array_equal(reloaded.ID, np.arange(EXPECTED_TEST_ROWS)):
    raise ValueError('Saved submission has invalid columns or IDs')
if not set(reloaded.TARGET).issubset({'rest', 'move'}):
    raise ValueError('Saved submission has invalid labels')
if any(name == 'eeg_comp.classical' or name.startswith('sklearn.linear_model')
       or name.startswith('sklearn.discriminant_analysis') or name.startswith('sklearn.svm')
       for name in sys.modules):
    raise RuntimeError('Classical estimator module imported in pure DL execution')
artifact_paths = [OUTPUT_DIR / name for name in ['metadata.csv', 'test_predictions.csv',
                  'submission.csv', 'epoch_selection.json', 'checkpoint_replay.json']]
artifact_paths += sorted(final_dir.glob('*')) + sorted(inner_dir.glob('*'))
manifest = {
    'competition': COMPETITION, 'competition_slug': SLUG, 'category': 'DL',
    'notebook': NOTEBOOK_NAME, 'build_id': BUILD_ID, 'source_sha256': SOURCE_HASHES,
    'seed': SEED, 'versions': VERSIONS, 'python_executable': sys.executable,
    'platform': platform.platform(), 'thread_limit': THREADS,
    'device': str(device), 'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
    'runtime': runtime_provenance(), 'model': asdict(config),
    'training': {**vars(args), 'max_inner_epochs': MAX_EPOCHS, 'cross_fixed_epochs': CROSS_EPOCHS},
    'epoch_selection': epoch_records, 'fits': fit_manifest,
    'information_budget': 'allowed labeled EEG for fitting; independent trial-local test inference',
    'confirmation_policy': 'Cross reserved people never scored or used for stopping; included only in final fitting',
    'threshold': .5, 'label_mapping': {'rest': 0, 'move': 1},
    'channels': list(CHANNELS), 'nominal_fs': FS, 'model_window': list(WINDOWS[WINDOW]),
    'normalization': 'runtime per trial per channel; no fitted state', 'filtering': 'none',
    'ordering': 'participant, numeric run, original marker order; zero-based IDs',
    'ordering_source': f'https://www.kaggle.com/competitions/{SLUG}/overview/evaluation',
    'rows': len(submission), 'id_min': 0, 'id_max': len(submission) - 1,
    'label_counts': submission.TARGET.value_counts().to_dict(),
    'raw_data_root': str(data_root), 'raw_inputs': input_files,
    'artifacts_sha256': {path.relative_to(OUTPUT_DIR).as_posix(): sha256_file(path) for path in artifact_paths},
    'checkpoint_replay': replay_records,
    'portability': 'Same-device replay verified. Historical L40S-to-CPU replay changed one thresholded Within prediction; cross-device identity is not assumed.',
    'elapsed_seconds': time.monotonic() - STARTED,
    'execution_context': ('kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') else 'local'),
    'status': 'raw-data training, inference and same-device replay completed; no upload performed',
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(submission.head())
print(f'Saved {len(submission)} rows to {OUTPUT_DIR / "submission.csv"}')
print(f'Elapsed: {manifest["elapsed_seconds"]:.1f} seconds')
